# 11 — Frequency-domain source comparison

Every emission model in the library, overlaid on a shared axis.  This
notebook is the interactive counterpart of `docs/comparison.md` — tweak
any parameter and the overlay re-renders live.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from harmonyemissions import Laser, Target, simulate
from harmonyemissions.units import keV_per_harmonic


def run_energy(model, laser, target):
    """Return (energy_keV, normalised spectrum) for overlay plots."""
    r = simulate(laser, target, model=model)
    spec = r.spectrum
    if 'photon_energy_keV' in spec.coords:
        E = np.asarray(spec.coords['photon_energy_keV'].values, dtype=float)
    else:
        n = np.asarray(spec.coords['harmonic'].values, dtype=float)
        E = n * keV_per_harmonic(laser.wavelength_um)
    S = np.asarray(spec.values, dtype=float)
    mask = (E > 0) & (S > 0)
    return E[mask], S[mask] / S[mask].max(), r

## Cross-source overlay

Every model at matched drive conditions on a shared photon-energy axis.

In [ ]:
runs = [
    ('BGP surface HHG',    'bgp', Laser(a0=10.0), Target.overdense(200.0, 0.05)),
    ('ROM surface HHG',    'rom', Laser(a0=10.0), Target.overdense(200.0, 0.05)),
    ('CSE nanobunching',   'cse', Laser(a0=15.0), Target.overdense(200.0, 0.01)),
    ('CWE (sub-relativistic)', 'cwe', Laser(a0=0.3), Target.overdense(400.0, 0.05)),
    ('Gas HHG (Ar)',       'lewenstein', Laser(a0=0.08, wavelength_um=0.8, duration_fs=20.0, polarization='linear'), Target.gas('Ar')),
    ('LWFA betatron',      'betatron', Laser(a0=2.0), Target.underdense(0.001, electron_energy_mev=500.0)),
    ('Inverse Compton',    'ics', Laser(a0=0.3, spot_fwhm_um=5.0, duration_fs=30.0), Target.electron_beam(beam_energy_mev=500.0)),
    ('Bremsstrahlung',     'bremsstrahlung', Laser(a0=5.0), Target.overdense(100.0)),
    ('Kα (Cu target)',     'kalpha', Laser(a0=5.0), Target.overdense(100.0, material='Cu')),
]
fig, ax = plt.subplots(figsize=(9, 5))
for label, model, laser, target in runs:
    E, S, _ = run_energy(model, laser, target)
    ax.loglog(E * 1e3, S, lw=1.1, label=label)
ax.set_xlabel('photon energy [eV]')
ax.set_ylabel('I / I_peak per source')
ax.set_xlim(0.1, 1e7)
ax.set_ylim(1e-6, 2.0)
ax.legend(ncol=2, fontsize=8, loc='lower left')
ax.grid(True, which='both', alpha=0.25)
ax.set_title('Frequency-domain overlay of every emission model')
plt.show()

## Surface-HHG regimes at the same a₀

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, model, target in [
    ('ROM / BGP',        'rom', Target.overdense(200.0, 0.05)),
    ('BGP universal',    'bgp', Target.overdense(200.0, 0.05)),
    ('CSE nanobunching', 'cse', Target.overdense(200.0, 0.01)),
    ('CWE',              'cwe', Target.overdense(400.0, 0.05)),
]:
    r = simulate(Laser(a0=10.0), target, model=model)
    n = r.spectrum.coords['harmonic'].values
    s = r.spectrum.values / r.spectrum.values.max()
    ax.loglog(n, s, lw=1.2, label=label)
ax.set_xlabel('harmonic order n')
ax.set_ylabel('I(n) / I_peak')
ax.set_title('Surface regimes at a₀ = 10')
ax.legend()
plt.show()

## γ-band overlay

In [ ]:
gamma_runs = [
    ('Betatron (500 MeV)',    'betatron', Laser(a0=2.0), Target.underdense(0.001, electron_energy_mev=500.0)),
    ('ICS (500 MeV beam)',    'ics', Laser(a0=0.3, spot_fwhm_um=5.0, duration_fs=30.0), Target.electron_beam(beam_energy_mev=500.0)),
    ('Bremsstrahlung (Wilks)', 'bremsstrahlung', Laser(a0=5.0), Target.overdense(100.0)),
    ('Kα (Ag target)',        'kalpha', Laser(a0=5.0), Target.overdense(100.0, material='Ag')),
]
fig, ax = plt.subplots(figsize=(8, 4.5))
for label, model, laser, target in gamma_runs:
    E, S, r = run_energy(model, laser, target)
    ax.loglog(E, S, lw=1.2, label=label)
ax.set_xlabel('photon energy [keV]')
ax.set_ylabel('I / I_peak')
ax.set_title('Hard-X-ray → γ-band sources')
ax.legend()
plt.show()

## Sweep: a₀ drives the surface-HHG cutoff

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for a0 in [3.0, 10.0, 24.0, 60.0]:
    r = simulate(Laser(a0=a0), Target.overdense(200.0, 0.05), model='rom')
    n = r.spectrum.coords['harmonic'].values
    s = r.spectrum.values / r.spectrum.values.max()
    ax.loglog(n, s, lw=1.1, label=f'a₀ = {a0:g}')
n_ref = np.geomspace(3, 3000, 40)
ax.loglog(n_ref, (n_ref / n_ref[0]) ** (-8 / 3), 'k:', lw=0.8, label='n⁻⁸/³')
ax.set_xlabel('harmonic order n')
ax.set_ylabel('I / I_peak')
ax.set_title('ROM spectrum vs a₀ — cutoff ≈ a₀³')
ax.legend()
plt.show()

## Sweep: beam energy in betatron / ICS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for E_mev in [500.0, 1000.0, 2000.0]:
    r = simulate(Laser(a0=2.0), Target.underdense(0.001, electron_energy_mev=E_mev), model='betatron')
    E, S = r.spectrum.coords['photon_energy_keV'].values, r.spectrum.values
    axes[0].loglog(E, S / S.max(), lw=1.1,
                    label=f'γE = {E_mev:.0f} MeV  (χ_e = {r.diagnostics["chi_e"]:.2e})')
axes[0].set_xlabel('photon energy [keV]')
axes[0].set_ylabel('I / I_peak')
axes[0].set_title('Betatron')
axes[0].legend(fontsize=8)
for E_mev in [100.0, 500.0, 1000.0]:
    r = simulate(Laser(a0=0.3, spot_fwhm_um=5.0, duration_fs=30.0),
                 Target.electron_beam(beam_energy_mev=E_mev),
                 model='ics')
    E, S = r.spectrum.coords['photon_energy_keV'].values, r.spectrum.values
    axes[1].loglog(E, S / S.max(), lw=1.1,
                    label=f'beam = {E_mev:.0f} MeV  (χ_e = {r.diagnostics["chi_e_nominal"]:.2e})')
axes[1].set_xlabel('photon energy [keV]')
axes[1].set_title('Inverse Compton')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Sweep: Kα target material

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for material in ['Al', 'Ti', 'Fe', 'Cu', 'Mo', 'Ag']:
    r = simulate(Laser(a0=5.0), Target.overdense(100.0, material=material), model='kalpha')
    E = r.spectrum.coords['photon_energy_keV'].values
    S = r.spectrum.values
    ax.semilogy(E, S / S.max(), lw=1.0, label=material)
ax.set_xlim(1.0, 30.0)
ax.set_ylim(1e-6, 2.0)
ax.set_xlabel('photon energy [keV]')
ax.set_ylabel('line intensity (norm.)')
ax.set_title('Kα/Kβ across target materials')
ax.legend()
plt.show()

See [`docs/comparison.md`](../docs/comparison.md) for the matching static
reference page and the decision matrix for picking the right source.